<a href="https://colab.research.google.com/github/1900690/YOLO-dataset-sprit/blob/main/AI_dataset_sprit_advance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#@title 1. データのアップロードと解凍
#@markdown FastLabelなどでエクスポートしたZIPファイルをアップロードしてください。

import os
import shutil
from google.colab import files

# すでにフォルダがある場合は初期化
if os.path.exists("/content/datasets"):
    shutil.rmtree("/content/datasets")

print("ZIPファイルをアップロードしてください:")
uploaded = files.upload()

if uploaded:
    file_name = list(uploaded.keys())[0]
    print(f"{file_name} を解凍しています...")
    # データを解凍
    shutil.unpack_archive(f'/content/{file_name}', '/content/')
    # アップロードしたZIPを削除
    os.remove(f'/content/{file_name}')
    print("データの解凍が完了しました！")
else:
    print("ファイルがアップロードされませんでした。")

ZIPファイルをアップロードしてください:


Saving eggplant-bbox_20260702171507.zip to eggplant-bbox_20260702171507.zip
eggplant-bbox_20260702171507.zip を解凍しています...
データの解凍が完了しました！


In [3]:
#@title 1.5. 画像とアノテーションの回転修正（オプション）
#@markdown 解凍されたデータの中に、向きが回転してしまっている画像がある場合、ここで修正できます。
#@markdown <br>※右に90度回転している画像を正常に戻すには、**「270」**（時計回りに270度＝反時計回りに90度）を選択してください。

修正する角度 = 270 #@param [0, 90, 180, 270] {type:"raw"}
対象の条件 = "指定した縦横ピクセルの場合のみ" #@param ["すべてのファイル", "ファイル名に特定の文字を含む場合のみ", "指定した縦横ピクセルの場合のみ"]
特定の文字 = "_rotated" #@param {type:"string"}
対象の横ピクセル_幅 = 3264 #@param {type:"integer"}
対象の縦ピクセル_高さ = 2448 #@param {type:"integer"}

import cv2
import numpy as np
import os
import glob

def rotate_image_and_bbox(image, bboxes, angle):
    """画像とYOLOフォーマット(正規化座標)のバウンディングボックスを回転する関数"""
    h, w = image.shape[:2]
    rotated_bboxes = []

    if angle == 90:
        rotated_image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
        for bbox in bboxes:
            cls, x, y, bw, bh = bbox
            rotated_bboxes.append([cls, 1.0 - y, x, bh, bw])
    elif angle == 180:
        rotated_image = cv2.rotate(image, cv2.ROTATE_180)
        for bbox in bboxes:
            cls, x, y, bw, bh = bbox
            rotated_bboxes.append([cls, 1.0 - x, 1.0 - y, bw, bh])
    elif angle == 270:
        rotated_image = cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
        for bbox in bboxes:
            cls, x, y, bw, bh = bbox
            rotated_bboxes.append([cls, y, 1.0 - x, bh, bw])
    else:
        rotated_image = image
        rotated_bboxes = bboxes

    return rotated_image, rotated_bboxes

originals = '/content/original'
annotations = '/content/yolo/annotations'

if 修正する角度 != 0 and os.path.exists(originals):
    print(f"回転修正処理を開始します (設定角度: {修正する角度}度)...")
    img_files = glob.glob(os.path.join(originals, "*"))
    processed_count = 0

    for img_path in img_files:
        basename = os.path.basename(img_path)
        filename, ext = os.path.splitext(basename)

        # ファイル名によるフィルタリング条件チェック
        if 対象の条件 == "ファイル名に特定の文字を含む場合のみ" and 特定の文字 not in filename:
            continue

        # 画像読み込み
        image = cv2.imread(img_path)
        if image is None:
            continue

        # 縦横ピクセルによるフィルタリング条件チェック
        if 対象の条件 == "指定した縦横ピクセルの場合のみ":
            h, w = image.shape[:2]
            if w != 対象の横ピクセル_幅 or h != 対象の縦ピクセル_高さ:
                continue

        lbl_path = os.path.join(annotations, filename + ".txt")

        # ラベル(YOLO)読み込み
        bboxes = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        bboxes.append([int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])])

        # 回転処理の実行
        corrected_image, corrected_bboxes = rotate_image_and_bbox(image, bboxes, 修正する角度)

        # 元ファイルに上書き保存
        cv2.imwrite(img_path, corrected_image)
        if os.path.exists(lbl_path) or len(corrected_bboxes) > 0:
            with open(lbl_path, 'w') as f:
                for bbox in corrected_bboxes:
                    f.write(f"{int(bbox[0])} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f} {bbox[4]:.6f}\n")

        processed_count += 1

    print(f"回転修正が完了しました。対象ファイル数: {processed_count}枚")
else:
    print("回転修正はスキップされました（角度が0、またはデータが存在しません）。")

回転修正処理を開始します (設定角度: 270度)...
回転修正が完了しました。対象ファイル数: 30枚


In [6]:
#@title 2. 画像の分割 (Train / Valid / Test)
#@markdown 全体から検証用(Valid)とテスト用(Test)に割り当てる割合をスライダーで設定してください。（残りが自動的に学習用になります）

validの割合 = 0.25 #@param {type:"slider", min:0, max:1, step:0.05}
testの割合 = 0.05 #@param {type:"slider", min:0, max:1, step:0.05}

import shutil
import os
from sklearn.model_selection import train_test_split
import math

# パスを作成
originals = '/content/original'
annotations = '/content/yolo/annotations'

# 分割先のパスを作成
dirs = [
    '/content/datasets/train/images', '/content/datasets/train/labels',
    '/content/datasets/valid/images', '/content/datasets/valid/labels',
    '/content/datasets/test/images', '/content/datasets/test/labels'
]

# フォルダを初期化・作成
if os.path.exists('/content/datasets/'):
    shutil.rmtree('/content/datasets/')
for d in dirs:
    os.makedirs(d, exist_ok=True)

# フォルダの中のファイルのリストを作成
read_files_annotations = sorted(os.listdir(annotations))
read_files_originals = sorted(os.listdir(originals))

total_files = len(read_files_annotations)
if total_files == 0:
    print("エラー: アノテーションファイルが見つかりません。")
else:
    # 全体に対する各データの件数を計算
    valid_count = int(total_files * validの割合)
    test_count = int(total_files * testの割合)

    # まずValidを分割
    if valid_count > 0:
        ann_rem, ann_valid, orig_rem, orig_valid = train_test_split(
            read_files_annotations, read_files_originals, test_size=valid_count, random_state=42
        )
    else:
        ann_rem, ann_valid, orig_rem, orig_valid = read_files_annotations, [], read_files_originals, []

    # 残りからTestを分割（残りがTestの件数より多い場合のみ）
    if test_count > 0 and len(ann_rem) > test_count:
        ann_train, ann_test, orig_train, orig_test = train_test_split(
            ann_rem, orig_rem, test_size=test_count, random_state=42
        )
    else:
        ann_train, ann_test, orig_train, orig_test = ann_rem, [], orig_rem, []

    # コピー実行用の関数
    def copy_files(file_list, src_dir, dst_dir):
        for f in file_list:
            shutil.copy(os.path.join(src_dir, f), dst_dir)

    # 画像とアノテーションをフォルダに振り分け
    copy_files(ann_train, annotations, '/content/datasets/train/labels')
    copy_files(ann_valid, annotations, '/content/datasets/valid/labels')
    copy_files(ann_test, annotations, '/content/datasets/test/labels')

    copy_files(orig_train, originals, '/content/datasets/train/images')
    copy_files(orig_valid, originals, '/content/datasets/valid/images')
    copy_files(orig_test, originals, '/content/datasets/test/images')

    # 最大公約数を計算 (TrainとValidの枚数から)
    train_len = len(orig_train)
    valid_len = len(orig_valid)
    gcd_val = math.gcd(train_len, valid_len) if train_len > 0 and valid_len > 0 else "計算不可"

    print("=== 分割結果 ===")
    print(f"学習用 (Train): {train_len}枚")
    print(f"検証用 (Valid): {valid_len}枚")
    print(f"テスト用 (Test): {len(orig_test)}枚")
    print(f"最大公約数: {gcd_val} (※バッチサイズを決める際の目安になります)")

=== 分割結果 ===
学習用 (Train): 42枚
検証用 (Valid): 14枚
テスト用 (Test): 2枚
最大公約数: 14 (※バッチサイズを決める際の目安になります)


#学習データに関する詳細ファイルを作成

In [ ]:
#@title 3. 学習設定ファイル(data.yaml)の作成
#@markdown 検出したい項目（クラス名）を**カンマ区切り**で入力してください。数（nc）は自動計算されます。
#@markdown <br>例： `OK, NG` や `犬, 猫, 鳥`

クラス名 = "fruit, flower" #@param {type:"string"}

import yaml

# 入力された文字列をカンマで分割し、前後の空白を削除してリスト化
names_list = [name.strip() for name in クラス名.split(",")]
nc = len(names_list)

# yamlに書き込むデータ構造
yaml_data = {
    'train': '../train/images',
    'val': '../valid/images',
    'test': '../test/images',
    'nc': nc,
    'names': names_list
}

# yamlファイルの書き出し
with open('/content/datasets/data.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print("以下の内容で data.yaml を作成しました：")
print("-" * 20)
print(f"クラス数 (nc): {nc}")
print(f"クラス名 (names): {names_list}")

In [ ]:
#@title 4. 完了したデータのダウンロード
#@markdown チェックを入れて実行すると、自動的にZIPファイルがダウンロードされます。
ダウンロードを実行する = True #@param {type:"boolean"}

import shutil
from google.colab import files

print("データセットをZIP化しています...")
shutil.make_archive('/content/datasets', 'zip', '/content/datasets')
print("ZIP化が完了しました。")

if ダウンロードを実行する:
    print("ダウンロードを開始します...")
    files.download("/content/datasets.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>